# RevalExo external error and domain-shift analysis

Inspect participant-level predictions and compare the internal and external acceleration-magnitude distributions. The 0.5 cutoff is descriptive only; no clinical threshold is selected here.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, roc_auc_score
P=Path('../data/processed')
pred=pd.read_csv(P/'revalexo_external_participant_predictions.csv')
pred['reference_prediction']=(pred.raw_probability>=.5).astype(int); pred['error_type']=np.where((pred.label_binary==pred.reference_prediction),'correct',np.where(pred.label_binary==1,'false_negative','false_positive'))
print('AUROC:',roc_auc_score(pred.label_binary,pred.raw_probability)); display(pred[['subject','group','raw_probability','error_type']].sort_values('raw_probability'))
print('Reference confusion matrix [[TN,FP],[FN,TP]]:'); print(confusion_matrix(pred.label_binary,pred.reference_prediction))
pred.to_csv(P/'revalexo_external_error_analysis.csv',index=False)

AUROC: 0.8714285714285714


,subject,group,raw_probability,error_type
0,Subject01_HC,HC,0.232114,correct
2,Subject03_HC,HC,0.444683,correct
8,Subject11_HC,HC,0.524870,false_positive
14,Subject24_HC,HC,0.568352,false_positive
1,Subject02_HC,HC,0.575779,false_positive
5,Subject07_ST,ST,0.641871,correct
6,Subject08_ST,ST,0.665832,correct
10,Subject15_ST,ST,0.675700,correct
16,Subject29_HC,HC,0.680326,false_positive
15,Subject26_ST,ST,0.683304,correct


Reference confusion matrix [[TN,FP],[FN,TP]]:
[[ 2  5]
 [ 0 10]]


In [2]:
internal=np.load(P/'validated_acceleration_magnitude_windows_float32.npy',mmap_mode='r'); raw=np.load(P/'revalexo_external_windows_float32.npy',mmap_mode='r')
ext=np.stack([np.linalg.norm(raw[:,:,0:3],axis=2),np.linalg.norm(raw[:,:,6:9],axis=2),np.linalg.norm(raw[:,:,12:15],axis=2)],axis=2)
rows=[]
for name,x in [('internal',internal),('revalexo',ext)]:
 q=np.percentile(np.asarray(x).reshape(-1,3),[1,25,50,75,99],axis=0)
 for i,placement in enumerate(['LB','RF','LF']): rows.append({'dataset':name,'placement':placement,'p01':q[0,i],'p25':q[1,i],'p50':q[2,i],'p75':q[3,i],'p99':q[4,i]})
shift=pd.DataFrame(rows); display(shift); shift.to_csv(P/'revalexo_external_feature_shift.csv',index=False)

,dataset,placement,p01,p25,p50,p75,p99
0,internal,LB,0.586061,0.912396,0.983641,1.075889,1.655266
1,internal,RF,0.742178,0.992094,1.052422,1.520594,4.467634
2,internal,LF,0.748753,1.000666,1.057340,1.532453,4.389260
3,revalexo,LB,0.732938,0.963386,1.003703,1.045276,1.463293
4,revalexo,RF,0.786948,1.012422,1.018426,1.153605,3.795561
5,revalexo,LF,0.827697,1.001888,1.009778,1.125181,3.557032


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
oof=pd.read_csv(P/'repeated_pooled_outer_predictions.csv')
cal=LogisticRegression(C=1e6,solver='lbfgs').fit(oof[['logit']],oof.label_binary)
ext=pred.copy(); ext['logit']=np.log(np.clip(ext.raw_probability,1e-6,1-1e-6)/np.clip(1-ext.raw_probability,1e-6,1))
ext['internal_oof_calibrated_probability']=cal.predict_proba(ext[['logit']])[:,1]
print('Internal OOF-calibrated external Brier:',brier_score_loss(ext.label_binary,ext.internal_oof_calibrated_probability))
display(ext[['subject','group','raw_probability','internal_oof_calibrated_probability']].sort_values('internal_oof_calibrated_probability'))
print('RevalExo individual age metadata available: no; age/cohort interpretation requires restricted metadata or new recruitment.')
ext.to_csv(P/'revalexo_external_calibrated_predictions.csv',index=False)

Internal OOF-calibrated external Brier: 0.19862658441459471


,subject,group,raw_probability,internal_oof_calibrated_probability
0,Subject01_HC,HC,0.232114,0.238750
2,Subject03_HC,HC,0.444683,0.535458
8,Subject11_HC,HC,0.524870,0.639208
14,Subject24_HC,HC,0.568352,0.691361
1,Subject02_HC,HC,0.575779,0.699943
5,Subject07_ST,ST,0.641871,0.771795
6,Subject08_ST,ST,0.665832,0.795753
10,Subject15_ST,ST,0.675700,0.805286
16,Subject29_HC,HC,0.680326,0.809687
15,Subject26_ST,ST,0.683304,0.812497


RevalExo individual age metadata available: no; age/cohort interpretation requires restricted metadata or new recruitment.
